In [6]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 1. Load environment variables (Make sure GROQ_API_KEY is in your .env)
load_dotenv()

# 2. Initialize the model using Groq instead of Google
model = init_chat_model(
    model="llama-3.1-8b-instant", 
    model_provider="groq"
)

# 3. Invoke the model
response = model.invoke("write me a 200 words paragraph on Artificial Intelligence")
print(response.content)

Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, and perception. These systems use complex algorithms and data analysis to simulate human-like intelligence, enabling them to adapt to new situations and improve their performance over time. AI has become increasingly prevalent in various industries, including healthcare, finance, transportation, and customer service, where it is used to automate tasks, enhance efficiency, and improve accuracy. For instance, AI-powered chatbots can handle customer inquiries, while AI-driven medical imaging systems can detect diseases more accurately than human radiologists. However, the rapid advancement of AI has also raised concerns about job displacement, bias, and accountability. As AI continues to evolve, researchers and developers are working to ensure that these systems are transparent, explainable, and ali

In [8]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

# 1. Load your .env file containing GROQ_API_KEY
load_dotenv()

# 2. Setup the agent with Groq and Summarization Middleware
agent = create_agent(
    model="groq:llama-3.1-8b-instant",  # Set the primary LLM to Groq
    checkpointer=InMemorySaver(),       # Keeps track of the chat history in memory
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.1-8b-instant",  # Use Groq to generate the summary too
            trigger=("messages", 10),           # Summarize once history hits 10 messages
            keep=("messages", 4)                # Keep the 4 most recent messages raw
        )
    ]
)

# 3. Configure the session thread (Required by the checkpointer)
config = {"configurable": {"thread_id": "coding_session_1"}}

# 4. Invoke the agent passing both the input state and the configuration
result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "Tell me a quick joke about coding."}]
    },
    config=config
)

# 5. Print the last message from the assistant
print(result["messages"][-1].content)

Why did the developer quit his job? 

Because he didn't get arrays.


In [10]:
from langchain_core.messages import HumanMessage

# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages Count: {len(response['messages'])}")

Messages Count: 4
Messages Count: 6
Messages Count: 8
Messages Count: 10
Messages Count: 6
Messages Count: 8


In [11]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

# 1. Load environment variables for GROQ_API_KEY
load_dotenv()

# 2. Define your tool
@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

# 3. Create the agent using Groq
agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.1-8b-instant",
            trigger=("tokens", 550),  # Triggers summary when context passes 550 tokens
            keep=("tokens", 200),     # Compresses everything down, leaving 200 tokens raw
        ),
    ]
)

# 4. Set configuration thread
config = {"configurable": {"thread_id": "hotel-test-session"}}

# Helper Token counter function
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

# 5. Run a quick test query to see it in action
result = agent.invoke(
    {"messages": [HumanMessage(content="Can you search for hotels in Paris?")]},
    config=config
)

print(f"Agent Reply:\n{result['messages'][-1].content}\n")
print(f"Total messages in history: {len(result['messages'])}")
print(f"Approximate total tokens: {count_tokens(result['messages'])}")

Agent Reply:
The function 'search_hotels' returned a long response to use more tokens but I used some of the information to describe the hotels.

Total messages in history: 4
Approximate total tokens: 85


In [15]:

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [16]:

agent=create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)


In [17]:

config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)


In [18]:

result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='482244c5-0843-4f05-ba1b-d4b48e31862d'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bjvc4c59p', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 309, 'total_tokens': 341, 'completion_time': 0.040962768, 'completion_tokens_details': None, 'prompt_time': 0.021200672, 'prompt_tokens_details': None, 'queue_time': 0.058516978, 'total_time': 0.06216344}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ee46d-bf25-74e0-91df-0cf64f694638-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body'

In [19]:

from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: Email sent successfully


In [20]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='482244c5-0843-4f05-ba1b-d4b48e31862d'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bjvc4c59p', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 309, 'total_tokens': 341, 'completion_time': 0.040962768, 'completion_tokens_details': None, 'prompt_time': 0.021200672, 'prompt_tokens_details': None, 'queue_time': 0.058516978, 'total_time': 0.06216344}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ee46d-bf25-74e0-91df-0cf64f694638-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body'

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [22]:

config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [23]:
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: 


In [24]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='9a242db9-facc-4e4f-b0de-af09f96ce6e4'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'jyhqpqvtk', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 309, 'total_tokens': 341, 'completion_time': 0.055187044, 'completion_tokens_details': None, 'prompt_time': 0.029926821, 'prompt_tokens_details': None, 'queue_time': 0.049140159, 'total_time': 0.085113865}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ee46f-0d3d-7990-9832-7eba5943d29f-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body

In [25]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [26]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [27]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='1dfa1d70-227d-4db2-8957-9f880e0667e7'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '1myyjh4pn', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 307, 'total_tokens': 337, 'completion_time': 0.026718731, 'completion_tokens_details': None, 'prompt_time': 0.019776915, 'prompt_tokens_details': None, 'queue_time': 0.046168705, 'total_time': 0.046495646}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ee470-3ddb-7c62-b85e-455afd1918db-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'Hello', 

In [28]:
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: 


In [29]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='1dfa1d70-227d-4db2-8957-9f880e0667e7'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '1myyjh4pn', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 307, 'total_tokens': 337, 'completion_time': 0.026718731, 'completion_tokens_details': None, 'prompt_time': 0.019776915, 'prompt_tokens_details': None, 'queue_time': 0.046168705, 'total_time': 0.046495646}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ee470-3ddb-7c62-b85e-455afd1918db-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'args